# Additional manoeuvre data: frozen NN pilots versus NN pilots

Acquire **10,000 new physical engagements**, with **20,000 recorded aircraft perspectives**.
Both F-16s use the validation-selected, post-tactical-adaptation NN checkpoints reported
in [07_tactical_adaptation.ipynb](07_tactical_adaptation.ipynb). There are no PPO updates.
Every engagement has a fresh continuous relative geometry and seed. Flights stop at
**300 in-game seconds or the first aircraft defeat**, including a crash. Both pilots'
51-feature observations and physical tracks are recorded at **10 Hz**, and their actual
selected skills/parameters at **1 Hz**. All outcomes are retained.

**Run All downloads the verified pilot archive, benchmarks real rollouts, collects the
campaign, verifies it, and commits/pushes incremental data backups to this branch with
Git LFS.** Set `COLLECT=False` and `PUBLISH=False` to inspect the setup first. Rerunning
resumes the same campaign; it never counts a second perspective as another flight.
The notebook is a long-running data acquisition job. Its benchmark estimates the actual
wall time and storage requirement before the main run. The original corpora remain usable.

Install the repository's dependencies in this kernel first: `%pip install -e ".[ml,video]"`.
Git and Git LFS must be installed, and `forecasting_ed_261104` must be checked out for
publication. Use `GIT_EXECUTABLE` below on Windows if Git is absent from the kernel PATH.
The companion script runs in fresh spawned processes on Windows and Linux.

In [ ]:
%%time
from pathlib import Path
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import time
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "scripts/generate_nn_duel_corpus.py").exists())
sys.path.insert(0, str(ROOT / "scripts"))
import generate_nn_duel_corpus as duel

FLIGHTS = int(os.getenv("BVR_NN_FLIGHTS", "10000"))  # Physical flights, not perspectives.
SEED = int(os.getenv("BVR_NN_SEED", "26110600"))
EPISODE_SECONDS = 300
WORK = Path(os.getenv("BVR_NN_WORK", str(ROOT / "artifacts/nn_duels_v1"))).resolve()
OUTPUT = Path(os.getenv("BVR_NN_OUTPUT", str(ROOT / "shared/corpora/nn-duels-v1"))).resolve()
DOWNLOAD_LFS = os.getenv("BVR_NN_DOWNLOAD", "1") == "1"
RUN_BENCHMARK = os.getenv("BVR_NN_BENCHMARK", "1") == "1"
COLLECT = os.getenv("BVR_NN_COLLECT", "1") == "1"
PUBLISH = os.getenv("BVR_NN_PUBLISH", "1") == "1"
GIT_EXECUTABLE = None  # e.g. r"C:\Program Files\Git\cmd\git.exe"
DETERMINISTIC = True  # False is a separate, seeded stochastic-policy campaign.
BUDGET_WORKERS = max(1, int((os.cpu_count() or 1) * .8))
WORKER_CANDIDATES = sorted({1, max(1, BUDGET_WORKERS//2), BUDGET_WORKERS})
BENCHMARK_FLIGHTS = 10  # One full-duration example from each scenario family.
WORKERS = BUDGET_WORKERS  # Replaced by the measured fastest candidate below.
BACKUP_EVERY = 100
SHARD_FLIGHTS = 100  # Immutable closed shards; keep fixed when resuming.
SCRIPT = ROOT / "scripts/generate_nn_duel_corpus.py"
WORK.mkdir(parents=True, exist_ok=True)

def run(command, *, work=WORK, output=OUTPUT, flights=FLIGHTS, seed=SEED,
        workers=WORKERS, limit=None):
    args = [sys.executable, "-u", str(SCRIPT), command, "--work", str(work),
            "--output", str(output), "--flights", str(flights), "--seed", str(seed),
            "--seconds", str(EPISODE_SECONDS), "--workers", str(workers),
            "--shard-flights", str(SHARD_FLIGHTS)]
    if limit is not None:
        args += ["--limit", str(limit)]
    if not DETERMINISTIC:
        args += ["--stochastic"]
    if command == "publish":
        args += ["--allow-partial"]  # Completeness is always explicit in manifest.json.
    env = os.environ.copy()
    if GIT_EXECUTABLE:
        env["PATH"] = str(Path(GIT_EXECUTABLE).parent) + os.pathsep + env.get("PATH", "")
    for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        env[key] = "1"
    subprocess.run(args, cwd=ROOT, env=env, check=True)

print(f"Target: {FLIGHTS:,} physical flights / {2*FLIGHTS:,} perspectives; <= {EPISODE_SECONDS}s")
print("Detected CPU count:", os.cpu_count(), "| worker candidates:", WORKER_CANDIDATES)
print("Work:", WORK, "| LFS publication:", OUTPUT)

## Frozen population and actual initial conditions

The source is the **5.7 MB `reproduction.zip` in `trained-pilot-episodes-v1`**,
verified against its publication manifest. It contains all ten *post*-tactical
checkpoints, their architectures/configuration and the source/assets used for that
corpus. The notebook-05 public checkpoint pool predates tactical adaptation and is
not substituted. Each source file and checkpoint hash is checked before collection;
source, software and configuration must match for resumption.

The old spawn adapter used a sampled axis, opposing courses and fixed speeds; nominal
scenario bearings/headings were not actually applied. Here the requested position,
heading and speed are passed directly into aircraft creation during reset, so JSBSim
starts in that geometry. The collector checks actual reset positions and velocity
vectors. North/West/Up mathematical angles are converted into the proper velocity
vectors; the original guidance interface retains its own heading convention.

Each continuous dimension uses stratified samples. Ten families cover mixed aspects,
left/right crossings, tail chase, cold/recommit, close merges, low energy climbs,
high/low encounters, offset support and defensive beam geometries. Horizontal
separation spans 6–65 km, altitude 1.5–11.5 km, and speed 200–380 m/s.
Initial velocities exclude the old corpus's exactly antiparallel pattern; all
requested states are unique. Novel geometry is not a guarantee of novel NN behaviour.
The 100 ordered pilot pairs, including self-pairs, each receive 100 fresh flights
in the default campaign. Roles and pair order are balanced without reusing geometry.

In [ ]:
%%time
if DOWNLOAD_LFS:
    # Reuse 09b's Windows-aware Git discovery, but download only the small pilot archive.
    sys.path.insert(0, str(ROOT))
    from bvr_behavior_prediction.paired_skill_forecasting import download_paired_corpora
    download_paired_corpora(ROOT, ["shared/corpora/trained-pilot-episodes-v1/reproduction.zip"],
                           git_executable=GIT_EXECUTABLE)
run("prepare")
config = duel.read(WORK / "config.json")
geometry = pd.DataFrame([{
    "family": s["family"], "pilot_A01": s["pilots"]["A01"], "pilot_B01": s["pilots"]["B01"],
    "horizontal_separation_m": np.linalg.norm(np.array(s["initial"]["B01"]["position"][:2]) -
                                             np.array(s["initial"]["A01"]["position"][:2])),
    "height_offset_m": s["initial"]["B01"]["position"][2] - s["initial"]["A01"]["position"][2],
    "heading_difference_deg": (s["initial"]["B01"]["heading_nwu_deg"] - s["initial"]["A01"]["heading_nwu_deg"]) % 360,
} for s in config["cases"]])
display(geometry.groupby("family").agg(flights=("family", "size"),
    minimum_range_m=("horizontal_separation_m", "min"), maximum_range_m=("horizontal_separation_m", "max"),
    minimum_height_offset_m=("height_offset_m", "min"), maximum_height_offset_m=("height_offset_m", "max")))
display(geometry.groupby(["pilot_A01", "pilot_B01"]).size().unstack(fill_value=0))
display(pd.DataFrame(config["pilots"]).T)

## Throughput benchmark and acceleration

Rollouts use a persistent **process pool**, bypassing Python's GIL and isolating
JSBSim state and global RNGs. Each worker loads each small frozen NN once, keeps it
in evaluation/inference mode and uses one PyTorch/BLAS thread. Nested thread pools
would compete with simulator workers. Same-pilot self-play batches both aircraft
in one NN forward pass. Guidance is recomputed every physics step, while NN actions
are chosen every second using the actual two-second history. Unused scripted expert
actions and reward calculations are skipped; flight dynamics, sensors, missiles,
controls and observation semantics retain the frozen implementation. No videos or
ACMI replays are generated. Arrays are compressed into bounded Parquet shards.

CPU FP32 inference avoids accelerator transfer and process/CUDA-context overhead
for these small networks. A centralized GPU inference server batching many worlds
could help a much larger deployment, but requires an additional benchmark and is
not claimed as an implemented speedup. Physics cadence stays at 0.1 s; the default
JSBSim inner integration step is preserved. The old simulator **forces full
throttle**, including during speed-labelled skills; this behaviour is recorded and
preserved to avoid silently changing the trained pilots' environment.

The benchmark below uses identical ten-case, 300-second-limited suites for every
worker count, in separate scratch campaigns. It saves wall time, simulated duration,
compressed bytes and a checksum signature of the actual data. Deterministic signatures
must agree across worker counts. Choose the measured fastest count; more processes
are not automatically faster. A cached benchmark is reused only for unchanged code
and software. The main campaign remains separate from benchmark flights.

In [ ]:
%%time
benchmark_path = WORK / "benchmark.json"
benchmark_identity = {"collector_hashes": config["collector_hashes"], "software": config["software"],
                      "workers": WORKER_CANDIDATES, "flights": BENCHMARK_FLIGHTS,
                      "seconds": EPISODE_SECONDS, "deterministic": DETERMINISTIC}
if RUN_BENCHMARK:
    cached = duel.read(benchmark_path) if benchmark_path.exists() else None
    if cached and cached["identity"] == benchmark_identity:
        benchmark = pd.DataFrame(cached["results"])
    else:
        rows = []
        for workers in WORKER_CANDIDATES:
            bench_id = duel.sha256(SCRIPT)[:12]
            bench_work = WORK / "benchmarks" / bench_id / f"workers_{workers}"
            start = time.perf_counter()
            run("collect", work=bench_work, flights=BENCHMARK_FLIGHTS, seed=SEED+100000,
                workers=workers)
            summaries = [duel.completed(bench_work, s) for s in duel.read(bench_work / "config.json")["cases"]]
            # Use persisted timing when resuming a completed benchmark.
            progress = duel.read(bench_work / "progress.json")
            elapsed = progress["wall_seconds"]
            signatures = sorted((s["episode_id"], sorted(s["files"].items())) for s in summaries)
            import hashlib
            rows.append({"workers": workers, "wall_seconds": elapsed,
                         "flights_per_hour": BENCHMARK_FLIGHTS*3600/elapsed,
                         "mean_game_seconds": np.mean([s["duration_s"] for s in summaries]),
                         "bytes_per_flight": sum(p.stat().st_size for p in (bench_work / "episodes").rglob("*.parquet"))/BENCHMARK_FLIGHTS,
                         "data_signature": hashlib.sha256(json.dumps(signatures).encode()).hexdigest()})
        benchmark = pd.DataFrame(rows)
        if DETERMINISTIC and benchmark.data_signature.nunique() != 1:
            raise RuntimeError("Worker counts produced different deterministic data; investigate before collection")
        duel.write(benchmark_path, {"identity": benchmark_identity, "results": rows})
    WORKERS = int(benchmark.loc[benchmark.flights_per_hour.idxmax(), "workers"])
    display(benchmark.drop(columns="data_signature"))
    best = benchmark.loc[benchmark.flights_per_hour.idxmax()]
    estimate_hours = FLIGHTS / best.flights_per_hour
    data_gib = FLIGHTS * best.bytes_per_flight / 2**30
    free_gib = shutil.disk_usage(WORK).free / 2**30
    print(f"Selected {WORKERS} workers; estimated acquisition {estimate_hours:.1f} hours before packaging/upload overhead.")
    print(f"Estimated compressed corpus {data_gib:.2f} GiB; work, publication and LFS objects roughly triple it; free {free_gib:.2f} GiB.")
    if free_gib < 3.5 * data_gib:
        raise RuntimeError("Insufficient free disk for work, publication and LFS object storage")
else:
    print("Benchmark disabled. Explicit worker count:", WORKERS)

## Collection, restart and off-machine backups

Completed episodes have six checksummed Parquet files (three per perspective) and
an atomic, fsynced completion marker. An interrupted flight is rerun with the same
seed; successful flights are verified and skipped. A campaign lock prevents two
kernels writing the same work directory. After a hard crash, inspect the PID in
`collection.lock` and remove that file only once the process has stopped.

Every 100 new flights the notebook closes immutable shards, verifies published data,
and, when `PUBLISH=True`, commits **only the corpus directory** and pushes this branch.
Git's pre-push hook uploads LFS payloads; publication checks the remote commit afterward.
Partial backups have `complete=false` and explicit physical-flight counts. The final
manifest can say `complete=true` only at 10,000 flights. A failed push raises immediately;
local episode checkpoints remain available to rerun the cell. Unrelated staged changes
are rejected by publication. Existing committed branch work will accompany a push.

Keep the work directory until the final package is verified and its upload is confirmed.
LFS storage and network transfer grow with the measured corpus size. Publication uses
bounded shards below 1 GB, not a single giant file; it does not rewrite old corpora.

In [ ]:
%%time
if COLLECT:
    while True:
        config = duel.read(WORK / "config.json")
        count = sum(duel.completed(WORK, s) is not None for s in config["cases"])
        if count < FLIGHTS:
            run("collect", workers=WORKERS, limit=min(BACKUP_EVERY, FLIGHTS-count))
        run("package")
        # Verify each materialized snapshot before any upload.
        run("verify")
        manifest = duel.read(OUTPUT / "manifest.json")
        print(f"Durable snapshot: {manifest['physical_flights']:,}/{FLIGHTS:,} physical flights; "
              f"{manifest['perspective_episodes']:,} perspectives; complete={manifest['complete']}")
        if PUBLISH:
            run("publish")
        if manifest["complete"]:
            break
    duel.verify(OUTPUT, require_complete=True)
else:
    run("status")
    print("Collection disabled; no new flights were requested by this execution.")

## Data contract for a later 09b-style experiment

`A01/` stores A01's observation of NN target B01; `B01/` stores B01's observation of
NN target A01. Each has `observations`, `tracks` and privileged `target_decisions`.
`episodes.parquet` identifies target/observer pilots and checkpoint hashes, actual
initial state, shared `scenario_group`, duration and outcome. Parquet row groups are
whole episodes. Time zero and the terminal sample are retained. A terminal sample
has no newly issued decision; future labels are never extended past the final action.

The helper below returns one perspective in the same column contract used by `09b`.
For the A01-observer direction only, its copy of the decision ID is canonicalized to
`A01`, the target name required by `09b.forecast_targets`; `actual_aircraft_id` preserves
physical identity. Observation axes and values are unchanged. This enables the same
**54 causal features**, **51-frame histories plus five acceleration lookback frames**,
first origin at **6 s**, and exact **+1/+2/+5 s** targets with independent missing masks.
The existing `09b.prepare_dataset` hardcodes its old corpus; it does not automatically
consume this new release. A later preparation pass must index these shards explicitly.

Keep **both views of each engagement in the same split**. Fit normalization only on
training histories. Decisions, pilot/checkpoint identity, future tracks and outcomes
remain outside encoder inputs. Combining two observations into 108 inputs is a
separate omniscient task unless both streams are available at inference time.
Reserve validation before using skill-gap feedback to tune scenario distributions;
actively selected follow-up data belongs in training, with a separate fixed evaluation
suite. Catalogue labels can share executed control code and are not physical-skill proof.

In [ ]:
%%time
if (OUTPUT / "manifest.json").exists():
    manifest = duel.verify(OUTPUT)
    episodes = pd.read_parquet(OUTPUT / "episodes.parquet")
    display(episodes.groupby(["observer_id", "family"]).agg(
        perspectives=("episode_id", "size"), mean_duration_s=("duration_s", "mean")))
    if len(episodes):
        episode = episodes.iloc[0]
        observations, tracks, target_decisions = duel.load_forecasting_view(
            OUTPUT, episode.episode_id, episode.observer_id)
        from bvr_behavior_prediction.paired_skill_forecasting import trained_examples
        classes = duel.read(WORK / "tactical_snapshot/training/pilot_001/config.json")["skill_names"]
        features, origins, targets = trained_examples(observations, tracks, target_decisions, classes)
        display(pd.Series({"physical_flights": manifest["physical_flights"],
            "perspectives": manifest["perspective_episodes"], "example_feature_shape": features.shape,
            "forecast_origins": len(origins), "valid_labels_per_horizon": (targets >= 0).sum(axis=0).tolist()}))
        display(target_decisions.head())
else:
    print("No collection snapshot yet.")

## Where might the missing skills appear? Hypotheses to test

The saved pre-split survey in `09b` on this branch examined 3,000 NN-versus-scripted
flights. **`commit`, `dive_defense`, and `drag_missile` had zero eligible episodes at
all three forecast horizons.** `accelerate_to_speed` and `climb_to_altitude` had only
**two episodes** each; `lock_target` had four. These are archived observations, not a
promise that a larger corpus will make those classes appear. The new audit below
uses the same six-second eligibility and exact-horizon rules and retains zero classes.

* **Energy/altitude gaps:** low-energy starts with a higher opponent may encourage
  acceleration and climbing; inverted height advantages may encourage descent and
  vertical defence. The low-energy/climb and high/low families test this. Full-throttle
  bypass can make acceleration choices physically indistinguishable; changing it
  requires a separately versioned environment and likely new pilot adaptation.
* **Commit/recommit:** cold, crossing and tail-chase starts place targets behind or
  outside the initial radar cone and may create search, lock and re-engagement
  transitions. Offset/support starts vary missile support geometry. Prolonging flights
  allows more post-shot/recommit phases, but first defeat still terminates the episode.
* **Missile defence gaps:** close merges and defensive beam starts may elicit earlier
  opposing launches and change time-to-impact and aspect, possibly yielding dive,
  drag or last-ditch responses. Recording incoming missiles and remaining weapons
  helps distinguish lack of threat exposure from refusal to select a defensive skill.
* **Adaptive next batch (speculation, not implemented):** after an initial development
  batch, relate eligible skill counts to family, energy gap, aspect, range, pilot pair
  and missile threat exposure. Sample undercovered cells and perturb promising starts;
  use Bayesian optimization or a cross-entropy search over initial conditions with
  rare-skill episode support as its score. Keep a broad random component, log sampling
  probabilities and seeds, and avoid success-only filtering. Count independent scenario
  groups and transitions, not just thousands of overlapping windows.
* **Policy support limits:** a frozen deterministic pilot may never choose a catalogue
  entry anywhere reachable. Inspect its logits on diverse histories first. A separately
  labelled stochastic-policy campaign (`DETERMINISTIC=False`, fresh work/output) can
  sample lower-probability choices while preserving NN control; it changes the behaviour
  distribution. If the missing skills have essentially zero probability, use curriculum
  fine-tuning or skill-conditioned demonstrators in a separate study. Forced catalogue
  actions would not demonstrate that these frozen pilots independently chose the skill.

Scenario novelty, catalogue coverage, physical behaviour and forecast learnability
are four distinct measurements. Report both sides and pilot-specific counts before
claiming a gap has been filled; a zero may persist despite all 10,000 new flights.

In [ ]:
%%time
if (OUTPUT / "skill_support.parquet").exists():
    support = pd.read_parquet(OUTPUT / "skill_support.parquet")
    display(support.pivot(index="skill", columns="horizon_s", values="windows"))
    display(support.pivot(index="skill", columns="horizon_s", values="perspective_episodes"))
    print("Still missing at one or more horizons:")
    display(support[support.windows.eq(0)])
    breakdown = pd.read_parquet(OUTPUT / "skill_support_by_pilot_family.parquet")
    display(breakdown.groupby(["pilot_id", "horizon_s"]).agg(supported_skills=("skill", lambda s: s[breakdown.loc[s.index, "windows"].gt(0)].nunique())))
    print("These episode counts include two correlated perspectives per physical engagement; split by scenario_group.")
else:
    print("Skill support will be measured from actual selections after the first closed shard.")